# Data Cleaning & Data Quality Analysis

Goals:
- Standardize column names
- Investigate missing values
- Detect infinite values
- Find duplicate records
- Fix encoding issues
- Optimize memory usage

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
DATA_DIR = Path("../data/raw")

monday_path = DATA_DIR / "Monday-WorkingHours.pcap_ISCX.csv"

df = pd.read_csv(monday_path)

df.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,49188,4,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,49486,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [3]:
df.columns.tolist()[:15]

[' Destination Port',
 ' Flow Duration',
 ' Total Fwd Packets',
 ' Total Backward Packets',
 'Total Length of Fwd Packets',
 ' Total Length of Bwd Packets',
 ' Fwd Packet Length Max',
 ' Fwd Packet Length Min',
 ' Fwd Packet Length Mean',
 ' Fwd Packet Length Std',
 'Bwd Packet Length Max',
 ' Bwd Packet Length Min',
 ' Bwd Packet Length Mean',
 ' Bwd Packet Length Std',
 'Flow Bytes/s']

In [5]:
df.columns = df.columns.str.strip()
df.columns.tolist()[:15]

['Destination Port',
 'Flow Duration',
 'Total Fwd Packets',
 'Total Backward Packets',
 'Total Length of Fwd Packets',
 'Total Length of Bwd Packets',
 'Fwd Packet Length Max',
 'Fwd Packet Length Min',
 'Fwd Packet Length Mean',
 'Fwd Packet Length Std',
 'Bwd Packet Length Max',
 'Bwd Packet Length Min',
 'Bwd Packet Length Mean',
 'Bwd Packet Length Std',
 'Flow Bytes/s']

In [6]:
missing_values = df.isnull().sum()

missing_values[missing_values > 0]

Flow Bytes/s    64
dtype: int64

In [7]:
numeric_df = df.select_dtypes(include=[np.number])

inf_count = np.isinf(numeric_df).sum()

inf_count[inf_count > 0]

Flow Bytes/s      373
Flow Packets/s    437
dtype: int64

In [8]:
df[np.isinf(df["Flow Bytes/s"])][
    ["Flow Duration", "Flow Bytes/s", "Flow Packets/s"]
].head(10)

,Flow Duration,Flow Bytes/s,Flow Packets/s
33,0,inf,inf
77,0,inf,inf
80,0,inf,inf
92,0,inf,inf
97,0,inf,inf
103,0,inf,inf
105,0,inf,inf
114,0,inf,inf
207,0,inf,inf
927,0,inf,inf


In [10]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,49188,4,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,49486,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
529913,443,18738,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
529914,53,60797,2,2,80,156,40,40,40.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
529915,53,154,2,2,64,96,32,32,32.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
529916,53,155,2,2,80,144,40,40,40.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [18]:
numeric_df = df.select_dtypes(include=[np.number])

np.isinf(numeric_df).sum().sum()

df.isnull().sum()[df.isnull().sum() > 0]

Flow Bytes/s      437
Flow Packets/s    437
dtype: int64

In [21]:
memory_mb = df.memory_usage(deep=True).sum() / 1024**2

print(f"Memory usage: {memory_mb:.2f} MB")

Memory usage: 343.15 MB


In [22]:
duplicates = df.duplicated().sum()

print(f"Duplicate rows: {duplicates}")

Duplicate rows: 26935


In [23]:
duplicate_rows = df[df.duplicated()]

duplicate_rows.head(10)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
2,49188,1,2,0,12,0,6,6,6.0,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,49188,1,2,0,12,0,6,6,6.0,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
6,49486,1,2,0,12,0,6,6,6.0,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
7,49486,1,2,0,12,0,6,6,6.0,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
46,389,1,2,0,7,0,7,0,3.5,4.949747,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
55,389,1,2,0,7,0,7,0,3.5,4.949747,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
69,389,1,2,0,7,0,7,0,3.5,4.949747,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
76,49478,1,2,0,37,0,31,6,18.5,17.677670,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
78,49478,1,2,0,37,0,31,6,18.5,17.677670,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
79,49478,1,2,0,37,0,31,6,18.5,17.677670,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [24]:
df.to_csv(
    "../data/processed/monday_cleaned.csv",
    index=False
)